In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import joblib
datapath = "../../../desktop/quant/hist/aaplIntra.csv"

In [2]:
df = pd.read_csv(datapath).copy()

In [3]:
df

,Dates,Open,Close,High,Low,Volume,Number Ticks
0,7/1/25 9:30,206.665,206.915,207.08,206.600,1035492,1433
1,7/1/25 9:30,206.910,206.710,206.92,206.500,119487,721
2,7/1/25 9:30,206.730,206.810,206.95,206.695,95679,603
3,7/1/25 9:30,206.840,207.200,207.22,206.790,164543,948
4,7/1/25 9:30,207.200,207.115,207.24,206.980,123276,626
...,...,...,...,...,...,...,...
281104,##########,273.900,273.670,274.60,273.470,95766657,2970
281105,##########,273.670,273.670,273.67,273.670,0,1
281106,12/19/25 15:59,273.690,273.900,273.91,273.600,556667,1933
281107,12/19/25 15:59,273.900,273.670,274.60,273.470,95766657,2970


In [4]:
# 1-step ahead Close price target
df["y"] = df["Close"].shift(-1)

# simple lag features (percentage changes for better scaling)
for k in [1, 2, 3, 5]:
    df[f"ret_lag_{k}"] = df["Close"].pct_change(k)

# add current close as a feature
df["close_norm"] = df["Close"] / df["Close"].rolling(window=20).mean()

# drop last row and any NAs from lags
df = df.dropna()

feature_cols = [c for c in df.columns if c.startswith("ret_lag_") or c == "close_norm"]
X = df[feature_cols].values.astype("float32")
y = df["y"].values.astype("float32")

In [5]:
# train validation split (random 80/20)
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=42)

In [ ]:
# Ridge regression (L2 regularization)
model = Ridge(alpha=1e-4)

model.fit(X_train, y_train)

# Evaluate on training and validation sets
train_score = model.score(X_train, y_train)
val_score = model.score(X_val, y_val)

print(f"Training R² score: {train_score:.6f}")
print(f"Validation R² score: {val_score:.6f}")

# Calculate MSE for comparison with Keras version
from sklearn.metrics import mean_squared_error
train_mse = mean_squared_error(y_train, model.predict(X_train))
val_mse = mean_squared_error(y_val, model.predict(X_val))
print(f"Training MSE: {train_mse:.6f}")
print(f"Validation MSE: {val_mse:.6f}")

Training R² score: 0.000002
Validation R² score: -0.000096
Training MSE: 601.466614
Validation MSE: 602.812622


In [8]:
joblib.dump(model, 'models/regularizedLinear.joblib')

['models/regularizedLinear.joblib']